In [16]:
import pandas as pd
import numpy as np
import joblib
# Load the raw hourly data
# We use ../ because the notebook is in the 'notebooks' folder
df_hourly = pd.read_csv("../data/weather_raw.csv")
df_hourly['date'] = pd.to_datetime(df_hourly['date'])

print(f"Loaded {len(df_hourly)} hourly rows.")

Loaded 745152 hourly rows.


In [17]:
# Group by day ('D') and aggregate
df_daily = df_hourly.resample('D', on='date').agg({
    'temperature': ['max', 'min', 'mean'],
    'humidity': 'mean',
    'pressure': 'mean',
    'precipitation': 'sum',
    'wind_speed': 'max',
    'radiation': 'sum',  
    'cloud_cover': 'mean', 
    'vpd': 'max',         
    'gusts': 'max'
})

df_daily.columns = [
    'temp_max', 'temp_min', 'temp_mean', 'humidity_mean', 'pressure_mean', 
    'precip_total', 'wind_max', 'radiation_sum', 'cloud_cover_mean', 'vpd_max', 'gusts_max'
]
df_daily = df_daily.reset_index()

print(f"Resampled to {len(df_daily)} daily rows.")
df_daily.head()

Resampled to 31048 daily rows.


,date,temp_max,temp_min,temp_mean,humidity_mean,pressure_mean,precip_total,wind_max,radiation_sum,cloud_cover_mean,vpd_max,gusts_max
0,1940-01-01,14.173500,11.373500,12.652667,90.831925,996.299528,12.6,37.936897,1387.0,85.958333,0.225786,55.800000
1,1940-01-02,15.023499,11.473500,13.196417,88.245493,993.865920,2.6,30.886656,1543.0,91.875000,0.344325,33.120000
2,1940-01-03,14.923500,12.523499,13.748500,87.440959,993.889454,7.3,36.075523,2118.0,59.083333,0.308066,63.360000
3,1940-01-04,14.973500,11.923500,13.567250,85.735737,1001.264402,4.7,30.259275,2157.0,59.916667,0.370095,52.920000
4,1940-01-05,15.573500,13.523499,14.654750,92.488617,1009.132900,4.0,23.667091,1607.0,94.000000,0.194082,38.519997


In [18]:
# Target: Temperatura Máxima de amanhã
df_daily['target_next_day_max'] = df_daily['temp_max'].shift(-1)


df_daily['temp_max_lag_1'] = df_daily['temp_max'].shift(1)
df_daily['temp_max_lag_2'] = df_daily['temp_max'].shift(2) 
df_daily['temp_max_lag_3'] = df_daily['temp_max'].shift(3) 

df_daily['temp_max_mean_3d'] = df_daily['temp_max'].rolling(window=3).mean()
df_daily['temp_range'] = df_daily['temp_max'] - df_daily['temp_min']

df_daily['pressure_tendency'] = df_daily['pressure_mean'].diff()

df_daily['precip_accum_7d'] = df_daily['precip_total'].rolling(window=7).sum()

# 5. Seasonal Encoding (Meses em formato cíclico)
df_daily['month'] = df_daily['date'].dt.month
df_daily['sin_month'] = np.sin(2 * np.pi * df_daily['month'] / 12)
df_daily['cos_month'] = np.cos(2 * np.pi * df_daily['month'] / 12)

df_daily = df_daily.dropna()

# Guardar os dados 
df_daily.to_csv("../data/weather_daily.csv", index=False)
print("Novas features calculadas e dataset guardado!")

Novas features calculadas e dataset guardado!


In [19]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# 1. Selection of Features
features = [
    'temp_max', 'temp_min', 'temp_mean', 'humidity_mean', 'pressure_mean', 
    'precip_total', 'wind_max', 'temp_max_lag_1', 'temp_max_lag_2', 'temp_max_lag_3',
    'temp_max_mean_3d', 'temp_range', 'sin_month', 'cos_month',
    'radiation_sum', 'cloud_cover_mean', 'vpd_max', 'gusts_max', 
    'pressure_tendency', 'precip_accum_7d'
]

# 2. Scaling
scaler_features = MinMaxScaler(feature_range=(0, 1))
scaler_target = MinMaxScaler(feature_range=(0, 1))

# Scale features and target separately
scaled_features = scaler_features.fit_transform(df_daily[features])
scaled_target = scaler_target.fit_transform(df_daily[['target_next_day_max']])
# Save models
joblib.dump(scaler_features, '../models/scaler_features.joblib')
joblib.dump(scaler_target, '../models/scaler_target.joblib')

print("Scalers guardados com sucesso!")

# 3. Create Sequences 
def create_sequences(data, target, window_size=7):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size)])
        y.append(target[i + window_size])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_features, scaled_target, window_size=7)

print(f"Shape of X (Input): {X.shape}") 
print(f"Shape of y (Target): {y.shape}") 

Scalers guardados com sucesso!
Shape of X (Input): (31034, 7, 20)
Shape of y (Target): (31034, 1)


In [20]:
# 1. Define split points
total_samples = len(X)
train_end = int(total_samples * 0.8)
val_end = int(total_samples * 0.9)

# 2. Slice the arrays (No shuffling!)
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")   
print(f"Test set:       {X_test.shape}")  

# 3. Save the preprocessed tensors
# We use .npy format for efficiency with NumPy arrays
np.save("../data/X_train.npy", X_train)
np.save("../data/y_train.npy", y_train)
np.save("../data/X_val.npy", X_val)
np.save("../data/y_val.npy", y_val)
np.save("../data/X_test.npy", X_test)
np.save("../data/y_test.npy", y_test)

print("\nAll tensors saved to the 'data' folder. Preprocessing complete!")

Training set:   (24827, 7, 20)
Validation set: (3103, 7, 20)
Test set:       (3104, 7, 20)

All tensors saved to the 'data' folder. Preprocessing complete!
